# Data loading

In this section, we load the raw startups dataset and take an initial look at its structure.

In [14]:
import pandas as pd

# Chemin vers le fichier brut
DATA_PATH = "../data/startups_raw.csv"

# Chargement du dataset complet
df_raw = pd.read_csv(DATA_PATH)

# Aperçu rapide
display(df_raw.head())
print("\nShape df_raw:", df_raw.shape)
print("\nInfo df_raw:")
df_raw.info()

,permalink,name,homepage_url,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at
0,/organization/-fame,#fame,http://livfame.com,Media,10000000,operating,IND,16,Mumbai,Mumbai,1,NaN,2015-01-05,2015-01-05
1,/organization/-qounter,:Qounter,http://www.qounter.com,Application Platforms|Real Time|Social Network...,700000,operating,USA,DE,DE - Other,Delaware City,2,2014-09-04,2014-03-01,2014-10-14
2,/organization/-the-one-of-them-inc-,"(THE) ONE of THEM,Inc.",http://oneofthem.jp,Apps|Games|Mobile,3406878,operating,NaN,NaN,NaN,NaN,1,NaN,2014-01-30,2014-01-30
3,/organization/0-6-com,0-6.com,http://www.0-6.com,Curated Web,2000000,operating,CHN,22,Beijing,Beijing,1,2007-01-01,2008-03-19,2008-03-19
4,/organization/004-technologies,004 Technologies,http://004gmbh.de/en/004-interact,Software,-,operating,USA,IL,"Springfield, Illinois",Champaign,1,2010-01-01,2014-07-24,2014-07-24



Shape df_raw: (66368, 14)

Info df_raw:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66368 entries, 0 to 66367
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   permalink          66368 non-null  object
 1   name               66367 non-null  object
 2   homepage_url       61310 non-null  object
 3   category_list      63220 non-null  object
 4   funding_total_usd  66368 non-null  object
 5   status             66368 non-null  object
 6   country_code       59410 non-null  object
 7   state_code         57821 non-null  object
 8   region             58338 non-null  object
 9   city               58340 non-null  object
 10  funding_rounds     66368 non-null  int64 
 11  founded_at         51147 non-null  object
 12  first_funding_at   66344 non-null  object
 13  last_funding_at    66368 non-null  object
dtypes: int64(1), object(13)
memory usage: 7.1+ MB


## Filtering & target variable

In this section, we filter the statuses of interest and create the binary target variable `success`.

In [15]:
# Filtrer les statuts pertinents
status_keep = ["acquired", "ipo", "closed"]
df_model = df_raw[df_raw["status"].isin(status_keep)].copy()

# Création de la variable cible binaire
success_mapping = {"acquired": 1, "ipo": 1, "closed": 0}
df_model["success"] = df_model["status"].map(success_mapping)

print("Shape df_model (filtré):", df_model.shape)
print("\nDistribution de success (counts):")
print(df_model["success"].value_counts())

print("\nDistribution de success (pourcentages):")
print((df_model["success"].value_counts(normalize=True) * 100).round(2))

Shape df_model (filtré): (13334, 15)

Distribution de success (counts):
success
1    7096
0    6238
Name: count, dtype: int64

Distribution de success (pourcentages):
success
1    53.22
0    46.78
Name: proportion, dtype: float64


## Dates & missing values

In this section, we convert date columns and analyse missing values.

In [16]:
import numpy as np

# Conversion des colonnes de dates
for col in ["founded_at", "first_funding_at", "last_funding_at"]:
    if col in df_model.columns:
        df_model[col] = pd.to_datetime(df_model[col], errors="coerce")

# Analyse des valeurs manquantes (en %)
missing_pct = df_model.isna().mean().sort_values(ascending=False) * 100
print("Pourcentage de valeurs manquantes par colonne (df_model):\n")
print(missing_pct.round(2))

Pourcentage de valeurs manquantes par colonne (df_model):

founded_at           28.00
state_code           16.57
city                 16.14
region               16.14
country_code         14.93
homepage_url          9.13
category_list         8.14
first_funding_at      0.01
name                  0.01
permalink             0.00
funding_total_usd     0.00
status                0.00
funding_rounds        0.00
last_funding_at       0.00
success               0.00
dtype: float64


## Preliminary features

In this section, we compute a few basic derived features without yet integrating them into the final DataFrame.

In [17]:
# Calcul des features préliminaires sur les lignes où les dates sont présentes
age_at_first_funding = (df_model["first_funding_at"] - df_model["founded_at"]).dt.days
time_between_first_last = (df_model["last_funding_at"] - df_model["first_funding_at"]).dt.days

# Conversion funding_total_usd en numérique pour log1p
funding_numeric = pd.to_numeric(df_model["funding_total_usd"], errors="coerce")
log_funding_total = np.log1p(funding_numeric)

features_preview = pd.DataFrame({
    "age_at_first_funding": age_at_first_funding,
    "time_between_first_last": time_between_first_last,
    "log_funding_total": log_funding_total,
})

print("Aperçu des features préliminaires:")
display(features_preview.head(10))

Aperçu des features préliminaires:


,age_at_first_funding,time_between_first_last,log_funding_total
15,1111.0,0.0,15.424949
20,134.0,0.0,13.122365
23,-181.0,411.0,14.745705
31,47.0,28.0,14.038655
32,3719.0,0.0,17.370859
34,0.0,0.0,10.819798
47,NaN,0.0,14.403298
58,365.0,0.0,NaN
64,95.0,0.0,NaN
67,896.0,0.0,13.458837


## DataFrames preparation

In this section, we explicitly prepare `df_raw` and `df_model` for the rest of the analysis.

In [18]:
# Vérification des DataFrames principaux
print("df_raw.shape:", df_raw.shape)
print("df_model.shape:", df_model.shape)

print("\nDistribution de success dans df_model:")
print(df_model["success"].value_counts())

print("\nDistribution de success (en %):")
print((df_model["success"].value_counts(normalize=True) * 100).round(2))

df_raw.shape: (66368, 14)
df_model.shape: (13334, 15)

Distribution de success dans df_model:
success
1    7096
0    6238
Name: count, dtype: int64

Distribution de success (en %):
success
1    53.22
0    46.78
Name: proportion, dtype: float64


## Summary

In this last section, we recap the DataFrame shapes, the target distribution, and a summary of the preliminary features.

In [ ]:
print("Initial EDA summary")
print("--------------------")
print("df_raw.shape:", df_raw.shape)
print("df_model.shape:", df_model.shape)

print("\nDistribution of success (counts):")
print(df_model["success"].value_counts())

print("\nDistribution of success (percentages):")
print((df_model["success"].value_counts(normalize=True) * 100).round(2))

print("\nSummary of preliminary features (basic statistics):")
print(features_preview.describe())

Résumé EDA initiale
--------------------
df_raw.shape: (66368, 14)
df_model.shape: (13334, 15)

Distribution de success (counts):
success
1    7096
0    6238
Name: count, dtype: int64

Distribution de success (pourcentages):
success
1    53.22
0    46.78
Name: proportion, dtype: float64

Résumé des features préliminaires (statistiques de base):
       age_at_first_funding  time_between_first_last  log_funding_total
count           9598.000000             13332.000000       11143.000000
mean            1631.430923               441.487399          15.192404
std             3788.444433               869.874163           2.387477
min            -9518.000000                 0.000000           1.098612
25%              123.000000                 0.000000          13.815512
50%              550.000000                 0.000000          15.573369
75%             1726.750000               591.250000          16.906553
max            75589.000000             33628.000000          24.127110
